# Please use the code cells below as a skeleton for Lab Assignment 5

The code involves the setup of Ollama and initialization of Ollama with the LLama 3.2 model

In [1]:
!sudo apt update
!sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
97 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as re

In [2]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [3]:
!pip install ollama

In [4]:
!ollama pull llama3.2

In [5]:
import ollama

def ollama_response(prompt, model='llama3.2', **params):
  messages = [
      {
          'role': 'system',
          'content': 'You are a helpful assistant.'
      },
      {
          'role': 'user',
          'content': prompt
      }
  ]
  response = ollama.chat(model=model, messages=messages, **params)
  return response['message']['content']

# Example
answer = ollama_response('Why is the sky blue?')
print(answer)

The sky appears blue because of a phenomenon called Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described it in the late 19th century.

Here's what happens:

1. **Sunlight enters the Earth's atmosphere**: When sunlight enters our atmosphere, it consists of a spectrum of colors, including all the colors of the visible spectrum (red, orange, yellow, green, blue, indigo, and violet).
2. **Light encounters tiny molecules**: The sunlight encounters tiny molecules of gases such as nitrogen (N2) and oxygen (O2) in the atmosphere.
3. **Scattering occurs**: These tiny molecules scatter the light in all directions, but they scatter shorter (blue) wavelengths more than longer (red) wavelengths. This is known as Rayleigh scattering.
4. **Blue light is scattered in all directions**: As a result of the scattering, the blue light is distributed throughout the atmosphere, reaching our eyes from all parts of the sky.
5. **Our eyes perceive the blue color**: When we l

In [6]:
# YOUR CODE STARTS HERE

# Lab Assignment 5: Sentiment Analysis with Zero-Shot Prompting, Few-Shot Prompting, and Multiple LLMs

**Author:** Kyuhyeon Cho
**ASU ID:** 1237610792
**Date:** March 4, 2026

In [7]:
# ==========================================
# Code Cell 1: Import Required Libraries and Load Data
# ==========================================

# Import all necessary libraries for data processing, NLP, and interaction with LLMs using Ollama
# Import the necessary libraries for data manipulation and topic modeling
import pandas as pd
import numpy as np
import ollama # For interacting with the local LLM
import nltk # Standard NLP library
import re # For regular expressions/text cleaning
from collections import Counter
from google.colab import drive

# Mount Google Drive
# This will trigger an authorization prompt
drive.mount('/content/drive')

# Requirement: Load the dataset (LIMIT 2000 ROWS)
# Assuming the file is directly in your "My Drive" folder
file_path = '/content/drive/MyDrive/restaurant_reviews_az.csv'

try:
    df = pd.read_csv(file_path, nrows=2000)
    print(" File loaded successfully!")
except FileNotFoundError:
    print(" File not found. Please check that 'restaurant_reviews_az.csv' is in your main Google Drive folder.")

# Check the data import
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 File loaded successfully!


,review_id,user_id,business_id,stars,useful,funny,cool,text,date,Sentiment
0,IVS7do_HBzroiCiymNdxDg,fdFgZQQYQJeEAshH4lxSfQ,sGy67CpJctjeCWClWqonjA,3,1,1,0,"OK, the hype about having Hatch chili in your ...",1/27/2020 22:59,1
1,QP2pSzSqpJTMWOCuUuyXkQ,JBLWSXBTKFvJYYiM-FnCOQ,3w7NRntdQ9h0KwDsksIt5Q,5,1,1,1,Pandemic pit stop to have an ice cream.... onl...,4/19/2020 5:33,1
2,oK0cGYStgDOusZKz9B1qug,2_9fKnXChUjC5xArfF8BLg,OMnPtRGmbY8qH_wIILfYKA,5,1,0,0,I was lucky enough to go to the soft opening a...,2/29/2020 19:43,1
3,E_ABvFCNVLbfOgRg3Pv1KQ,9MExTQ76GSKhxSWnTS901g,V9XlikTxq0My4gE8LULsjw,5,0,0,0,I've gone to claim Jumpers all over the US and...,3/14/2020 21:47,1
4,Rd222CrrnXkXukR2iWj69g,LPxuausjvDN88uPr-Q4cQA,CA5BOxKRDPGJgdUQ8OUOpw,4,1,0,0,"If you haven't been to Maynard's kitchen, it'...",1/17/2020 20:32,1


In [8]:
# ==========================================
# Code Cell 2: Data Preprocessing
# ==========================================

# Remove reviews with 3 stars to create clear positive and negative categories
df_filtered = df[df['stars'] != 3]

# Create a dataset for this assignment by randomly selecting 50 positive reviews (4 or 5 stars)
positive_reviews = df_filtered[df_filtered['stars'] >= 4]
positive_sample = positive_reviews.sample(n=50, random_state=42)

# Create a dataset by randomly selecting 50 negative reviews (1 or 2 stars)
negative_reviews = df_filtered[df_filtered['stars'] <= 2]
negative_sample = negative_reviews.sample(n=50, random_state=42)

# Combine the positive and negative samples into a single dataset
df_subset = pd.concat([positive_sample, negative_sample])

# Optional but recommended: Shuffle the combined dataset so it's not strictly ordered
# (50 positive followed by 50 negative) and reset the index
df_subset = df_subset.sample(frac=1, random_state=42).reset_index(drop=True)

# Display the shape and value counts to verify we have exactly 50 of each
print(f"Total reviews in subset: {len(df_subset)}")
print(df_subset['stars'].apply(lambda x: 'Positive (4-5)' if x >= 4 else 'Negative (1-2)').value_counts())

Total reviews in subset: 100
stars
Negative (1-2)    50
Positive (4-5)    50
Name: count, dtype: int64


In [9]:
# ==========================================
# Code Cell 3: Perform Sentiment Analysis Using Zero-Shot Learning
# ==========================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm # To show a progress bar during classification

# Step 1: Create true labels for evaluation
# Map true stars to binary labels: 1 for Positive (4-5 stars), 0 for Negative (1-2 stars)
true_labels = [1 if stars >= 4 else 0 for stars in df_subset['stars']]
predicted_labels = []

print("Running Zero-Shot Classification with Llama 3.2...")

# Step 2: Predict sentiment labels for the selected 100 reviews without providing any labeled training examples
for review in tqdm(df_subset['text'], desc="Classifying reviews"):
    # Define a strict zero-shot prompt instructing the model to output ONLY the label
    prompt = f"""Classify the sentiment of the following restaurant review as either 'positive' or 'negative'.
    Return ONLY the word 'positive' or 'negative', with no other text, punctuation, or explanation.

    Review: "{review}"

    Sentiment:"""

    try:
        # Using the ollama_response function defined in the skeleton code
        response = ollama_response(prompt, model='llama3.2')

        # Clean the response to prevent formatting errors (lowercase, strip whitespace)
        cleaned_response = response.strip().lower()

        # Map the LLM's text output to our binary labels
        if 'positive' in cleaned_response:
            predicted_labels.append(1)
        elif 'negative' in cleaned_response:
            predicted_labels.append(0)
        else:
            # Fallback just in case the model is too chatty or returns something unexpected
            predicted_labels.append(0)

    except Exception as e:
        print(f"Error processing review: {e}")
        predicted_labels.append(0) # Fallback on error

# Step 3: Evaluate model performance using precision, recall, f1, and accuracy
accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels, average='binary')
recall = recall_score(true_labels, predicted_labels, average='binary')
f1 = f1_score(true_labels, predicted_labels, average='binary')

# Print out the evaluation metrics
print("\n--- Zero-Shot Performance Metrics (Llama 3.2) ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Running Zero-Shot Classification with Llama 3.2...


Classifying reviews: 100%|██████████| 100/100 [00:20<00:00,  4.85it/s]


--- Zero-Shot Performance Metrics (Llama 3.2) ---
Accuracy:  0.9500
Precision: 0.9787
Recall:    0.9200
F1 Score:  0.9485


In [10]:
# ==========================================
# Code Cell 4: Perform Sentiment Analysis Using Few-Shot Learning
# ==========================================

# Step 1: Select a few examples (2 positive and 2 negative reviews) for few-shot learning.
# We pull these from the remaining data outside our 100-review subset to avoid data leakage.
remaining_data = df_filtered[~df_filtered.index.isin(df_subset.index)]

few_shot_pos = remaining_data[remaining_data['stars'] >= 4].sample(n=2, random_state=42)
few_shot_neg = remaining_data[remaining_data['stars'] <= 2].sample(n=2, random_state=42)

# Extract the text for our prompt examples
pos_ex_1 = few_shot_pos.iloc[0]['text']
pos_ex_2 = few_shot_pos.iloc[1]['text']
neg_ex_1 = few_shot_neg.iloc[0]['text']
neg_ex_2 = few_shot_neg.iloc[1]['text']

# Step 2: Use these examples to guide the LLM in classifying sentiment for the selected 100 reviews
predicted_labels_few_shot = []

print("Running Few-Shot Classification with Llama 3.2...")

for review in tqdm(df_subset['text'], desc="Classifying reviews (Few-Shot)"):
    # Construct the Few-Shot Prompt with our selected examples
    prompt = f"""Classify the sentiment of the following restaurant review as either 'positive' or 'negative'.
    Return ONLY the word 'positive' or 'negative', with no other text, punctuation, or explanation.

    Here are some examples:
    Review: "{pos_ex_1}"
    Sentiment: positive

    Review: "{neg_ex_1}"
    Sentiment: negative

    Review: "{pos_ex_2}"
    Sentiment: positive

    Review: "{neg_ex_2}"
    Sentiment: negative

    Now classify this review:
    Review: "{review}"
    Sentiment:"""

    try:
        response = ollama_response(prompt, model='llama3.2')
        cleaned_response = response.strip().lower()

        # Map output to binary labels
        if 'positive' in cleaned_response:
            predicted_labels_few_shot.append(1)
        elif 'negative' in cleaned_response:
            predicted_labels_few_shot.append(0)
        else:
            predicted_labels_few_shot.append(0) # Fallback on unexpected formatting

    except Exception as e:
        print(f"Error processing review: {e}")
        predicted_labels_few_shot.append(0)

# Step 3: Evaluate model performance using precision, recall, f1, and accuracy
accuracy_fs = accuracy_score(true_labels, predicted_labels_few_shot)
precision_fs = precision_score(true_labels, predicted_labels_few_shot, average='binary')
recall_fs = recall_score(true_labels, predicted_labels_few_shot, average='binary')
f1_fs = f1_score(true_labels, predicted_labels_few_shot, average='binary')

# Print out the evaluation metrics
print("\n--- Few-Shot Performance Metrics (Llama 3.2) ---")
print(f"Accuracy:  {accuracy_fs:.4f}")
print(f"Precision: {precision_fs:.4f}")
print(f"Recall:    {recall_fs:.4f}")
print(f"F1 Score:  {f1_fs:.4f}")

Running Few-Shot Classification with Llama 3.2...


Classifying reviews (Few-Shot): 100%|██████████| 100/100 [00:20<00:00,  4.81it/s]


--- Few-Shot Performance Metrics (Llama 3.2) ---
Accuracy:  0.8700
Precision: 0.9744
Recall:    0.7600
F1 Score:  0.8539


In [11]:
# ==========================================
# Code Cell 5: Experiment with Multiple LLMs
# ==========================================

# Step 1: Pull the second LLM. Llama 3.2 was pulled in the skeleton code,
# so we will pull Mistral (a 7B parameter model) for our second LLM.
!ollama pull mistral

# Step 2: Select two distinct LLMs for sentiment analysis.
model_1 = 'llama3.2'
model_2 = 'mistral'

print(f"\nComparing {model_1} and {model_2} on a sample of 5 reviews...\n")

# We will grab 5 reviews from our subset to experiment and easily display the output
sample_reviews = df_subset.sample(n=5, random_state=42)

# Step 3: Design prompts to utilize both LLMs for sentiment analysis
for index, row in sample_reviews.iterrows():
    review_text = row['text']
    true_stars = row['stars']
    # Map the true stars for easy comparison visually
    true_sentiment = 'positive' if true_stars >= 4 else 'negative'

    # Prompt designed to be identical for both models to ensure a fair comparison
    prompt = f"""You are a sentiment analysis bot. Classify the following restaurant review as either 'positive' or 'negative'.
    Return ONLY the exact word 'positive' or 'negative' with no extra text or punctuation.

    Review: "{review_text}"

    Sentiment:"""

    # Run inference on both models
    try:
        # We use the ollama_response function defined in the skeleton code
        output_model_1 = ollama_response(prompt, model=model_1).strip().lower()
        output_model_2 = ollama_response(prompt, model=model_2).strip().lower()
    except Exception as e:
        output_model_1 = "Error"
        output_model_2 = "Error"
        print(f"Encountered error: {e}")

    # Step 4: Display the output
    print(f"Review Extract: \"{review_text[:120]}...\"")
    print(f"Actual Sentiment:  {true_sentiment} ({true_stars} stars)")
    print(f"{model_1.capitalize()} Prediction: {output_model_1}")
    print(f"{model_2.capitalize()} Prediction:  {output_model_2}")
    print("-" * 70)



Comparing llama3.2 and mistral on a sample of 5 reviews...

Review Extract: "Checked out Rockabilly on Monday. Brian Dean Trio was great. Dance floor has potential but too small. If they could remo..."
Actual Sentiment:  positive (4 stars)
Llama3.2 Prediction: negative
Mistral Prediction:  positive
----------------------------------------------------------------------
Review Extract: "Moved to Tucson about two years ago. First restaurant I visited was Prep & Pastry.  It was in a very charming non-fancy ..."
Actual Sentiment:  positive (5 stars)
Llama3.2 Prediction: positive
Mistral Prediction:  positive
----------------------------------------------------------------------
Review Extract: "Two times we attempted to eat in this place... waiting time is ridiculous, they said to wait for one hour and we decided..."
Actual Sentiment:  negative (1 stars)
Llama3.2 Prediction: negative
Mistral Prediction:  negative
----------------------------------------------------------------------
Revie

## Text Cell 6
## Acknowledgments

For the completion of this lab assignment, I utilized Google's Gemini to assist with writing the Python code structure, formatting the pandas data manipulation, designing the prompt templates for zero-shot and few-shot learning, and setting up the evaluation metrics (`sklearn.metrics`).

Additionally, as required by the assignment, I utilized the local LLMs Llama 3.2 and Mistral via Ollama for the sentiment analysis inference.

No other human collaboration was involved in the completion of these code cells.

## Text Cell 7
# Discussion and Observations

### 1. Zero-Shot vs. Few-Shot Learning Performance
**Comparison:**
* In my experiment, **Zero-Shot prompting** achieved an impressive accuracy of **0.9400 (94%)** and an F1 score of **0.9375**.
* Surprisingly, **Few-Shot prompting** decreased the overall performance, achieving an accuracy of **0.8700 (87%)** and an F1 score of **0.8539**.
* **Contrast:** While few-shot learning is typically expected to improve performance by anchoring the model's understanding, in this specific case, it caused a significant drop in **Recall** (from 0.9000 to 0.7600). This indicates that the few-shot model became overly strict and started misclassifying positive reviews as negative. This could be due to the specific random few-shot examples confusing the model, or the model over-fitting to specific words in the negative examples provided in the prompt.

### 2. Cases Where LLM Predictions Differ from Actual Labels
Upon reviewing the outputs from the multiple LLM comparison (Code Cell 5), there was a clear instance where a model's classification did not match the actual user rating:
* **Example:** A review stated, *"Chose this restaurant because we wanted Mexican food with a bar. We arrived after 3pm, it was pretty slow, as it was a M..."*. The actual label was **Positive (4 stars)**. However, the **Llama 3.2** model predicted it as **negative**. (Mistral correctly predicted it as positive).

### 3. Potential Reasons for Misclassifications
Analyzing the discrepancy above, the misclassification by Llama 3.2 likely stems from common NLP challenges with mixed sentiment:
* **Mixed Sentiments & Trigger Words:** The reviewer mentioned "it was pretty slow." While the reviewer was likely describing the crowd level (i.e., not busy) rather than complaining about slow service, Llama 3.2 likely latched onto the word "slow" as a negative semantic trigger, missing the broader, positive context of the 4-star experience.

### 4. Differences in Outputs from Various LLMs (Llama 3.2 vs. Mistral)
When comparing the Llama 3.2 model to the Mistral model on the 5-review sample subset:
* **Instruction Following:** Both models perfectly followed the output constraint to return *only* the exact word "positive" or "negative" without adding conversational filler or punctuation.
* **Nuance:** Mistral demonstrated a slightly better grasp of nuanced context in this small sample. While Llama 3.2 was tricked by the word "slow" in a 4-star review and flagged it as negative, Mistral correctly recognized the overall positive nature of the review. Overall, Mistral appeared slightly more robust against misleading keywords.

## Text Cell 8
## LLM-based Iterative Learning: Summary Task

### 1. The Prompt Used for ChatGPT
To complete the summary task, I used the following prompt with ChatGPT-4o to summarize the findings of my experiment:

> **Prompt:** "You are an expert data analyst. Please summarize the findings of my sentiment analysis experiment based on these constraints: Keep it under 4 sentences, mention both Llama 3.2 and Mistral, and include the actual accuracy metrics. Here is the data: Zero-shot Llama 3.2 achieved 94% accuracy. Few-shot Llama 3.2 dropped to 87% accuracy. On a 5-review sample, Mistral correctly classified a nuanced 4-star review mentioning 'slow' service, while Llama 3.2 falsely flagged it as negative."

### 2. Initial AI Response vs. My Original Response
My Initial Response: Zero-shot Llama 3.2 did better than few-shot (94% vs 87%). When testing Mistral and Llama 3.2, they both followed instructions well, but Mistral was better at reading a tricky review that used the word "slow", whereas Llama got confused and marked it negative.

AI Initial Response: Our sentiment analysis experiment revealed that Llama 3.2 achieved 94% accuracy in a zero-shot setting. However, its performance dropped to 87% when utilizing few-shot prompting. On a 5-review sample, Mistral correctly classified a tricky 4-star review mentioning "slow" service, whereas Llama 3.2 was misled by the negative keyword. Both models processed the data very quickly and efficiently.

### 3. Commentary and Observations
Comparing the two responses, I observed the following:

Strengths of the AI's output: The summary is professional, clearly states the metrics, and captures the core comparative finding between the two models much better than my initial draft.

Gaps/Weaknesses: The response was exactly 4 sentences long, hitting the absolute limit of my constraint, making it feel a bit choppy. It also added an unprompted detail about the models processing the data "quickly and efficiently," which was not part of the dataset I provided.

### 4. Iterative Refinement
To improve the output, I continued the dialogue with the AI to refine the summary.

Follow-up Prompt: "That is a great start, but please make it slightly more concise and cohesive (maximum 3 sentences) and remove the unprompted mention about processing speed."

AI's Revised Response: "Our sentiment analysis experiment revealed that Llama 3.2 achieved a robust 94% accuracy in a zero-shot setting, though its performance unexpectedly regressed to 87% when utilizing few-shot prompting. Despite Llama 3.2's strong overall metrics, comparative testing on a 5-review sample demonstrated that Mistral handles nuanced text better. Specifically, Mistral correctly classified a tricky 4-star review mentioning 'slow' service, whereas Llama 3.2 was misled by the negative keyword and falsely flagged it."

### 5. Final Perfected Version and Justification
Final Summary: Our sentiment analysis experiment revealed that Llama 3.2 achieved a robust 94% accuracy in a zero-shot setting, though its performance unexpectedly regressed to 87% when utilizing few-shot prompting. Despite Llama 3.2's strong overall metrics, comparative testing on a 5-review sample demonstrated that Mistral handles nuanced text better. Specifically, Mistral correctly classified a tricky 4-star review mentioning "slow" service, whereas Llama 3.2 was misled by the negative keyword and falsely flagged it.